# Patient Risk Stratification — Master Pipeline

End-to-end ML pipeline for 4-class patient risk prediction (LOW / MEDIUM / HIGH / CRITICAL).  
Run cells top-to-bottom; intermediate results are carried between steps automatically.

| # | Step | Output |
|---|------|--------|
| 1 | Feature Engineering & Feature Store | `TRAINING_FEATURES`, `TEST_FEATURES` tables |
| 1b | Hyperparameter Tuning (Ray Tune) *(optional)* | Best config in `HPO_RESULTS` table |
| 2 | Distributed Remote Training & Evaluation | Registered model version, promotion decision |
| 3 | Model Deployment | `PATIENT_RISK_SERVICE` REST endpoint |
| 4 | Model Monitoring | `PATIENT_RISK_MONITOR` drift monitor |

> **Steps 1b and 2 block** until their SPCS ML Jobs complete. Set `RUN_HPO = False` to skip tuning.

In [ ]:
%load_ext autoreload

In [ ]:
import json
import logging
import os
import sys

# Ensure project root is on sys.path so `source.*` imports resolve
_root = os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s — %(message)s",
    force=True,
)

from IPython.display import HTML, Markdown, display

print(f"Python        : {sys.version.split()[0]}")
print(f"Working dir   : {_root}")
print("Environment ready.")

In [ ]:
from source.configs import get_config
from source.utils import get_session

config  = get_config("source/config.yaml")
session = get_session(config.snowflake.connection_name)
session.use_database(config.snowflake.database)
session.use_schema(config.snowflake.schema_name)
session.use_warehouse(config.snowflake.warehouse)

print("Snowflake session ready.")
print(f"  Database     : {config.snowflake.database}")
print(f"  Schema       : {config.snowflake.schema_name}")
print(f"  Warehouse    : {config.snowflake.warehouse}")
print(f"  Compute pool : {config.compute.compute_pool}")
print(f"  Model name   : {config.model.model_name}")

---
## Step 1 — Feature Engineering & Feature Store

Initialises the Snowflake Feature Store, registers the `PATIENT` entity and `PATIENT_FEATURES`
feature view (auto-refreshes every minute), then writes `TRAINING_FEATURES` (80 %) and
`TEST_FEATURES` (20 %) tables used by downstream steps.

| Feature | Formula | Clinical meaning |
|---------|---------|-----------------|
| `SHOCK_INDEX` | `HEART_RATE / SYSTOLIC_BP` | Haemodynamic instability proxy |
| `PULSE_PRESSURE` | `SYSTOLIC_BP − DIASTOLIC_BP` | Cardiovascular health indicator |
| `BMI_CATEGORY` | Bucket of `BMI` | Weight classification |
| `VITAL_SIGNS_SEVERITY` | Abnormal-vital score 0–9 | Combined vitals alarm score |

In [ ]:
%autoreload
from source.pipeline.step1_feature_engineering import run as run_step1

step1_result = run_step1(config, session)

display(Markdown("**Step 1 complete ✅**"))
print(json.dumps(step1_result, indent=2, default=str))

---
## Step 1b — Hyperparameter Tuning (Ray Tune on SPCS)

Optional step that runs a **distributed Ray Tune search** on SPCS before the full training
run in Step 2. Each trial trains the model with a sampled hyperparameter configuration and
reports `f1_macro` to the Ray scheduler; ASHA prunes poor-performing trials early.

When complete, `train_hpo.py` writes the best configuration to the `HPO_RESULTS` table.
The `train.py` entrypoint in Step 2 reads this table automatically — no manual wiring needed.

| Parameter | Default | Notes |
|-----------|---------|-------|
| `HPO_NUM_SAMPLES` | 30 | Total trials (more = better search, slower) |
| `HPO_NUM_INSTANCES` | 3 | SPCS nodes: rank-0 = Ray head, 1+ = workers |
| `HPO_SEARCH_ALG` | `optuna` | `"random"` \| `"hyperopt"` \| `"optuna"` — requires `PYPI_ACCESS_INTEGRATION` for non-random |
| `HPO_SCHEDULER` | `asha` | `"asha"` \| `"pbt"` \| `"fifo"` |

> ⏳ **Blocks** until the Ray Tune job finishes. Wall-time ≈ `HPO_NUM_SAMPLES × avg_trial_time / HPO_NUM_INSTANCES`.  
> ⏭️ Set `RUN_HPO = False` to skip and let Step 2 use default hyperparameters.

In [ ]:
# ── HPO parameters ────────────────────────────────────────────────────────
RUN_HPO = True           # Set False to skip and use Step 2 defaults

# Search space — values are plain dicts; train_hpo.py reconstructs tune.* primitives
HPO_SEARCH_SPACE = {
    "n_estimators":     {"type": "randint",    "lower": 100,  "upper": 600},
    "max_depth":        {"type": "randint",    "lower": 3,    "upper": 12},
    "learning_rate":    {"type": "loguniform", "lower": 1e-4, "upper": 0.3},
    "subsample":        {"type": "uniform",    "lower": 0.6,  "upper": 1.0},
    "colsample_bytree": {"type": "uniform",    "lower": 0.6,  "upper": 1.0},
    "reg_alpha":        {"type": "loguniform", "lower": 1e-4, "upper": 10.0},
    "reg_lambda":       {"type": "loguniform", "lower": 1e-4, "upper": 10.0},
}

HPO_NUM_SAMPLES   = 30        # total Ray Tune trials
HPO_NUM_INSTANCES = 3         # SPCS nodes (1 head + 2 workers)
HPO_SEARCH_ALG    = "optuna"  # "random" | "hyperopt" | "optuna"
HPO_SCHEDULER     = "asha"    # "asha"   | "pbt"      | "fifo"

# External access integration created in 01_setup_infrastructure.ipynb.
# Grants the job container egress to PyPI so optuna/hyperopt can be pip-installed.
HPO_EXTERNAL_ACCESS_INTEGRATIONS = ["PYPI_ACCESS_INTEGRATION"]

In [ ]:
%autoreload
hpo_best_params = {}   # populated below; train.py reads HPO_RESULTS if this is non-empty

if not RUN_HPO:
    print("ℹ️  HPO skipped (RUN_HPO=False) — Step 2 will use default hyperparameters.")
    hpo_result = {"status": "skipped", "reason": "RUN_HPO=False"}
else:
    from source.framework.train import RayHPOConfig, RemoteTrainer

    db     = config.snowflake.database
    schema = config.snowflake.schema_name

    hpo_config = RayHPOConfig(
        search_space=HPO_SEARCH_SPACE,
        metric="f1_macro",
        mode="max",
        num_samples=HPO_NUM_SAMPLES,
        scheduler=HPO_SCHEDULER,
        search_alg=HPO_SEARCH_ALG,
    )

    trainer = RemoteTrainer(
        session=session,
        compute_pool=config.compute.compute_pool,
        stage=f"{db}.{schema}.JOB_PAYLOADS",
        source_dir="source",
    )

    print("Submitting Ray HPO job:")
    print(f"  Trials       : {HPO_NUM_SAMPLES}")
    print(f"  SPCS nodes   : {HPO_NUM_INSTANCES}  (rank-0 = Ray head, 1+ = workers)")
    print(f"  Algorithm    : {HPO_SEARCH_ALG} / {HPO_SCHEDULER}")
    print(f"  Search space : {list(HPO_SEARCH_SPACE.keys())}")
    print(f"  Pip packages : {hpo_config.pip_packages}")

    hpo_job = trainer.submit_hpo(
        hpo_config=hpo_config,
        entrypoint="train_hpo.py",
        num_instances=HPO_NUM_INSTANCES,
        external_access_integrations=HPO_EXTERNAL_ACCESS_INTEGRATIONS,
    )

    trainer.wait_and_log(hpo_job)

    # Read best params written by train_hpo.py to the HPO_RESULTS table
    # Expected schema: BEST_PARAMS (VARCHAR/JSON), BEST_SCORE (FLOAT), CREATED_AT (TIMESTAMP)
    results_table = f"{db}.{schema}.HPO_RESULTS"
    try:
        best_row = session.sql(
            f"SELECT BEST_PARAMS, BEST_SCORE "
            f"FROM {results_table} ORDER BY CREATED_AT DESC LIMIT 1"
        ).to_pandas()

        if not best_row.empty:
            hpo_best_params = json.loads(best_row.iloc[0]["BEST_PARAMS"])
            best_score      = best_row.iloc[0].get("BEST_SCORE")
            score_str       = f"{best_score:.4f}" if isinstance(best_score, float) else str(best_score)
            display(Markdown(f"**Best `f1_macro`: `{score_str}`**"))
            print("Best hyperparameters:")
            print(json.dumps(hpo_best_params, indent=2))
        else:
            print(f"⚠️  {results_table} is empty — check train_hpo.py logs above.")

    except Exception as exc:
        print(f"⚠️  Could not read {results_table}: {exc}")
        print("    Step 2 train.py will fall back to default hyperparameters.")

    hpo_result = {
        "status": "success",
        "job_id": hpo_job.id,
        "num_trials": HPO_NUM_SAMPLES,
        "search_alg": HPO_SEARCH_ALG,
        "best_params": hpo_best_params,
    }
    display(Markdown("**Step 1b complete ✅ — best params written to `HPO_RESULTS`, Step 2 will pick them up automatically.**"))

---
## Step 2 — Distributed Remote Training & Evaluation

Submits a training job to SPCS via **Snowflake ML Jobs**.  
`NUM_INSTANCES > 1` provisions multiple nodes; each node receives `RANK` and `WORLD_SIZE`
environment variables for coordinated distributed training.

After the job completes the model version is evaluated against `TEST_FEATURES` and checked
against the **promotion gate** (accuracy ≥ 0.80, f1_macro ≥ 0.75) before Step 3 can proceed.

> ⏳ **This cell blocks** until the ML Job finishes. Logs stream to output below.

In [ ]:
# ── Training parameters ───────────────────────────────────────────────────
# NUM_INSTANCES controls distributed training node count.
#   1  → single-node remote training on SPCS
#   3  → distributed across 3 nodes (demo default — each runs at its own RANK)
NUM_INSTANCES = 3

In [ ]:
%autoreload
from source.pipeline.step2_train_evaluate import run as run_step2

step2_result = run_step2(config, session, num_instances=NUM_INSTANCES)

version_name   = step2_result["version_name"]
should_promote = step2_result["should_promote"]

gate_label = "PASSED ✅" if should_promote else "FAILED ❌"
display(Markdown(
    f"**Step 2 complete — Promotion gate: {gate_label}**  \n"
    f"Model version: `{version_name}`"
))
print(json.dumps(step2_result, indent=2, default=str))

---
## Step 3 — Model Deployment

Deploys the promoted model version as a scalable SPCS REST inference service
(`PATIENT_RISK_SERVICE`). If an existing service is running it is dropped and re-created
to ensure a clean state.

A sample inference against three rows from `TEST_FEATURES` is fired to confirm the
endpoint is healthy before proceeding.

> ⏭️ Skipped automatically if the Step 2 promotion gate failed.

In [ ]:
if not should_promote:
    print(
        "⚠️  Promotion gate FAILED — skipping deployment.\n"
        "   Review Step 2 output for which metrics fell below threshold.\n"
        "   Re-run Step 2 after fixing the training configuration."
    )
    step3_result = {"status": "skipped", "reason": "promotion_gate_failed"}
else:
    from source.pipeline.step3_deploy import run as run_step3

    step3_result = run_step3(config, session, version_name=version_name)

    display(Markdown("**Step 3 complete ✅**"))
    print(json.dumps(step3_result, indent=2, default=str))

---
## Step 4 — Model Monitoring

Registers a **Snowflake Model Monitor** (`PATIENT_RISK_MONITOR`) for the deployed version.

The monitor compares:
- **Feature distribution** of `STREAMING_PATIENT_DATA` (live predictions) against `TEST_FEATURES` baseline
- **Prediction distribution** drift across the four risk classes

Alerts fire automatically when drift exceeds configured thresholds.

> ⏭️ Skipped automatically if Step 3 was skipped.

In [ ]:
if step3_result.get("status") == "skipped":
    print("⚠️  Skipping monitoring — deployment was not completed.")
    step4_result = {"status": "skipped", "reason": "deployment_skipped"}
else:
    from source.pipeline.step4_monitor import run as run_step4

    step4_result = run_step4(config, session, version_name=version_name)

    display(Markdown("**Step 4 complete ✅**"))
    print(json.dumps(step4_result, indent=2, default=str))

---
## Pipeline Run Summary

In [ ]:
def _fmt(val, precision=3):
    return f"{val:.{precision}f}" if isinstance(val, float) else (str(val) if val is not None else "—")

steps_summary = [
    ("1",  "Feature Engineering",       step1_result),
    ("1b", "Hyperparameter Tuning",     hpo_result),
    ("2",  "Training & Evaluation",     step2_result),
    ("3",  "Deployment",                step3_result),
    ("4",  "Monitoring",                step4_result),
]

rows = []
for num, name, result in steps_summary:
    status = result.get("status", "unknown")
    icon   = {"success": "✅", "skipped": "⏭️"}.get(status, "❌")

    if num == "1":
        detail = (
            f"training_rows={result.get('training_rows', '—')}, "
            f"test_rows={result.get('test_rows', '—')}"
        )
    elif num == "1b":
        bp = result.get("best_params", {})
        if bp:
            lr  = bp.get("learning_rate")
            lr_str = f"{lr:.4f}" if isinstance(lr, float) else str(lr) if lr else "—"
            detail = (
                f"trials={result.get('num_trials', '—')}, "
                f"alg={result.get('search_alg', '—')}, "
                f"best_lr={lr_str}, "
                f"best_n_estimators={bp.get('n_estimators', '—')}"
            )
        else:
            detail = result.get("reason", "no best_params available")
    elif num == "2":
        m      = result.get("metrics", {})
        passed = "PASSED" if result.get("should_promote") else "FAILED"
        detail = (
            f"accuracy={_fmt(m.get('accuracy'))}, "
            f"f1_macro={_fmt(m.get('f1_macro'))}, "
            f"promotion={passed}, version={result.get('version_name', '—')}"
        )
    elif num == "3":
        detail = (
            f"service={result.get('service_name', '—')}, "
            f"version={result.get('version_name', '—')}"
        )
    elif num == "4":
        detail = (
            f"monitor={result.get('monitor_name', '—')}, "
            f"monitor_status={result.get('monitor_status', '—')}"
        )
    else:
        detail = result.get("reason", "")

    rows.append(
        f"<tr>"
        f"<td style='padding:8px 14px;font-size:18px'>{icon}</td>"
        f"<td style='padding:8px 14px'><b>Step {num}</b> — {name}</td>"
        f"<td style='padding:8px 14px'><code>{status}</code></td>"
        f"<td style='padding:8px 14px;color:#888'>{detail}</td>"
        f"</tr>"
    )

display(HTML(f"""
<table style="border-collapse:collapse;width:100%;font-family:monospace;font-size:13px">
  <thead>
    <tr style="background:#0E2A47;color:white">
      <th style="padding:10px 14px"></th>
      <th style="padding:10px 14px;text-align:left">Step</th>
      <th style="padding:10px 14px;text-align:left">Status</th>
      <th style="padding:10px 14px;text-align:left">Details</th>
    </tr>
  </thead>
  <tbody>{"".join(rows)}</tbody>
</table>
"""))